In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.models import vgg19
from torchvision.transforms import ToTensor
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import io
import sys
import random
import os
from pathlib import Path

plt.switch_backend('Agg')

# Dataset paths - relative to notebook location
PROJECT_ROOT = Path.cwd()
DATASET_ROOT = PROJECT_ROOT / "datasets"
TRAIN_LR_PATH = DATASET_ROOT / "DIV2K_train_LR_bicubic" / "X4"
VALID_LR_PATH = DATASET_ROOT / "DIV2K_valid_LR_bicubic" / "X4"

SCALE_FACTOR = 4
BATCH_SIZE = 8
LR_SIZE = 96
HR_SIZE = LR_SIZE * SCALE_FACTOR
TRAIN_SAMPLES = 800
EPOCHS = 30

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch Version: {torch.__version__}")
print(f"Device: {device}")
print("-" * 50)


In [ ]:
def psnr(y_true, y_pred):
    mse = torch.mean((y_true - y_pred) ** 2)
    return 20 * torch.log10(1.0 / torch.sqrt(mse + 1e-10))

def ssim(y_true, y_pred):
    """SSIM metric that handles both batched and single tensors."""
    # For batched tensors [B, C, H, W], compute per-sample SSIM then average
    # For single tensors [C, H, W], compute SSIM directly
    if len(y_true.shape) == 4:  # Batched: [B, C, H, W]
        mu1 = y_true.mean(dim=[1, 2, 3], keepdim=True)
        mu2 = y_pred.mean(dim=[1, 2, 3], keepdim=True)
        sigma1_sq = ((y_true - mu1) ** 2).mean(dim=[1, 2, 3], keepdim=True)
        sigma2_sq = ((y_pred - mu2) ** 2).mean(dim=[1, 2, 3], keepdim=True)
        sigma12 = ((y_true - mu1) * (y_pred - mu2)).mean(dim=[1, 2, 3], keepdim=True)
    else:  # Single sample: [C, H, W]
        mu1, mu2 = y_true.mean(), y_pred.mean()
        sigma1_sq = ((y_true - mu1) ** 2).mean()
        sigma2_sq = ((y_pred - mu2) ** 2).mean()
        sigma12 = ((y_true - mu1) * (y_pred - mu2)).mean()
    
    c1, c2 = 0.01**2, 0.03**2
    ssim_val = ((2*mu1*mu2 + c1) * (2*sigma12 + c2)) / ((mu1**2 + mu2**2 + c1) * (sigma1_sq + sigma2_sq + c2))
    return ssim_val.mean() if len(y_true.shape) == 4 else ssim_val


In [ ]:
# ----------------------------------------------------------------------
# 2. REAL-WORLD DEGRADATION (GPU Optimized)
# ----------------------------------------------------------------------

def get_gaussian_kernel_fast(kernel_size=7, sigma=1.5, device='cpu'):
    """Fast Gaussian kernel generation with fixed size."""
    ax = torch.arange(-(kernel_size//2), kernel_size//2 + 1, dtype=torch.float32, device=device)
    xx, yy = torch.meshgrid(ax, ax, indexing='ij')
    kernel = torch.exp(-(xx**2 + yy**2) / (2.0 * sigma**2))
    kernel = kernel / torch.sum(kernel)
    kernel = kernel.view(1, 1, kernel_size, kernel_size)
    return kernel

def apply_blur(img):
    """Apply Gaussian blur with varying sigma."""
    sigma = random.uniform(0.5, 2.0)
    kernel = get_gaussian_kernel_fast(kernel_size=7, sigma=sigma, device=img.device)
    
    # Apply depthwise convolution for each channel
    img = img.unsqueeze(0)  # Add batch dimension if not present
    if img.shape[1] == 1:
        img = img.repeat(1, 3, 1, 1)
    
    channels = []
    for i in range(img.shape[1]):
        channel = img[:, i:i+1, :, :]
        blurred = F.conv2d(channel, kernel, padding=3)
        channels.append(blurred)
    
    out = torch.cat(channels, dim=1)
    return out.squeeze(0) if out.shape[0] == 1 else out

def apply_random_resize(img):
    """Simplified resize operation - GPU optimized."""
    h, w = img.shape[-2:]
    scale = random.uniform(0.75, 1.25)
    nh = max(4, int(h * scale))
    nw = max(4, int(w * scale))
    
    img_resized = F.interpolate(img.unsqueeze(0), size=(nh, nw), mode='bilinear', align_corners=False)
    img = F.interpolate(img_resized, size=(h, w), mode='bilinear', align_corners=False)
    return img.squeeze(0)

def apply_noise(img):
    """Apply Gaussian noise."""
    std = random.uniform(0.0, 0.03)
    noise = torch.randn_like(img) * std
    return torch.clamp(img + noise, 0.0, 1.0)

def apply_jpeg(img):
    """JPEG compression - only applied probabilistically."""
    if random.random() < 0.5:
        # Convert to PIL Image for JPEG compression
        img_np = img.permute(1, 2, 0).cpu().numpy()
        img_np = np.clip(img_np * 255, 0, 255).astype(np.uint8)
        pil_img = Image.fromarray(img_np)
        
        # Apply JPEG compression
        quality = random.choice([70, 80, 90])
        buffer = io.BytesIO()
        pil_img.save(buffer, format='JPEG', quality=quality)
        buffer.seek(0)
        compressed = Image.open(buffer)
        
        # Convert back to tensor
        img_np = np.array(compressed).astype(np.float32) / 255.0
        img = torch.from_numpy(img_np).permute(2, 0, 1).to(img.device)
    
    return img

def degrade_once(hr_img):
    x = apply_blur(hr_img)
    x = apply_random_resize(x)
    x = apply_noise(x)
    x = apply_jpeg(x)
    return x

def degrade_twice(hr_img):
    x = degrade_once(hr_img)
    x = degrade_once(x)
    return x

def generate_synthetic_lr(hr_img, scale=SCALE_FACTOR):
    """Apply second-order degradation at HR size then downscale to LR."""
    degraded = degrade_twice(hr_img)
    h = degraded.shape[-2] // scale
    w = degraded.shape[-1] // scale
    lr = F.interpolate(degraded.unsqueeze(0), size=(h, w), mode='bilinear', align_corners=False)
    return lr.squeeze(0)


In [ ]:
# ----------------------------------------------------------------------
# 3. DATASET LOADING AND AUGMENTATION
# ----------------------------------------------------------------------

class DIV2KDataset(Dataset):
    """Custom Dataset for DIV2K with augmentation and synthetic degradation."""
    def __init__(self, num_samples=TRAIN_SAMPLES, lr_size=LR_SIZE, hr_size=HR_SIZE, scale=SCALE_FACTOR, 
                 data_path=None, transform=None):
        self.num_samples = num_samples
        self.lr_size = lr_size
        self.hr_size = hr_size
        self.scale = scale
        self.transform = transform
        self.images = []
        
        # Try to load DIV2K dataset from project folder
        data_path = data_path or TRAIN_LR_PATH
        try:
            print(f"Attempting to load DIV2K dataset from {data_path}...")
            if data_path.exists():
                # Load all PNG images from the folder
                image_files = sorted(list(data_path.glob("*.png")))[:num_samples]
                print(f"Found {len(image_files)} images in dataset folder.")
                
                if len(image_files) > 0:
                    for img_path in image_files:
                        img = Image.open(img_path).convert('RGB')
                        img_tensor = ToTensor()(img)  # Converts to [0,1] range and [C, H, W]
                        # Note: These are LR images. We'll use them and generate HR via upscaling for training
                        # The degradation pipeline will generate synthetic LR from HR patches
                        self.images.append(img_tensor)
                    print(f"Successfully loaded {len(self.images)} DIV2K images.")
                else:
                    raise FileNotFoundError(f"No PNG images found in {data_path}")
            else:
                raise FileNotFoundError(f"Dataset path does not exist: {data_path}")
        except Exception as e:
            print(f"DIV2K Loading failed: {e}. Using simulated data.")
            print(f"Creating synthetic DIV2K-like dataset...")
            self.images = [torch.rand(3, hr_size, hr_size) for _ in range(num_samples)]
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img = self.images[idx]  # This could be LR or HR depending on what was loaded
        
        # If the loaded image is smaller than HR_SIZE, it's likely an LR image
        # Upscale it to HR_SIZE for training purposes
        if img.shape[1] < self.hr_size or img.shape[2] < self.hr_size:
            # Upscale LR to HR size using bilinear interpolation
            img = F.interpolate(img.unsqueeze(0), size=(self.hr_size, self.hr_size), 
                               mode='bilinear', align_corners=False).squeeze(0)
        
        hr_img = img
        
        # Random crop to HR_SIZE
        if hr_img.shape[1] > self.hr_size or hr_img.shape[2] > self.hr_size:
            top = random.randint(0, hr_img.shape[1] - self.hr_size)
            left = random.randint(0, hr_img.shape[2] - self.hr_size)
            # Align to scale factor
            top = (top // self.scale) * self.scale
            left = (left // self.scale) * self.scale
            hr_img = hr_img[:, top:top+self.hr_size, left:left+self.hr_size]
        
        # Generate synthetic LR from HR patch using second-order degradation
        lr_patch = generate_synthetic_lr(hr_img, scale=self.scale)
        
        # Apply augmentation to both LR and HR together (matching TensorFlow version)
        if random.random() < 0.5:
            lr_patch = torch.flip(lr_patch, dims=[2])
            hr_img = torch.flip(hr_img, dims=[2])
        
        if random.random() < 0.5:
            k = random.randint(0, 3)
            lr_patch = torch.rot90(lr_patch, k, dims=[1, 2])
            hr_img = torch.rot90(hr_img, k, dims=[1, 2])
        
        return lr_patch, hr_img

def load_or_simulate_dataset(num_samples, batch_size, lr_shape, hr_shape):
    """
    Loads the real DIV2K dataset and applies preprocessing and augmentation.
    """
    try:
        print("Attempting to load real DIV2K dataset...")
        
        # In practice, you would download and load DIV2K images from disk
        # For now, we simulate with synthetic data
        dataset = DIV2KDataset(num_samples=num_samples, lr_size=lr_shape[0], hr_size=hr_shape[0])
        
        # Apply preprocessing, shuffling, and batching
        # Note: num_workers=0 because __getitem__ uses CUDA operations which don't work well with multiprocessing
        # GPU optimization: pin_memory=True for faster GPU transfer
        dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, 
                               num_workers=0, pin_memory=True)
        
        print(f"TFDS DIV2K dataset processing configured with BATCH_SIZE={batch_size}.")
        return dataloader
    
    except Exception as e:
        print(f"TFDS Loading failed: {e}. Falling back to simulation.")
        print(f"Creating SIMULATED DIV2K dataset: {num_samples} samples, batch size {batch_size}")
        dataset = DIV2KDataset(num_samples=num_samples, lr_size=lr_shape[0], hr_size=hr_shape[0])
        dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, 
                               num_workers=0, pin_memory=True)
        return dataloader


In [ ]:
# ----------------------------------------------------------------------
# 3.5. TEST DATASET (BSD100) - LR generated via bicubic downsampling
# ----------------------------------------------------------------------

class BSD100Dataset(Dataset):
    """Custom Dataset for BSD100 test set."""
    def __init__(self, scale=SCALE_FACTOR, data_path=None, transform=None):
        self.scale = scale
        self.transform = transform
        self.images = []
        
        # Try to load BSD100 dataset from project folder (using validation set)
        data_path = data_path or VALID_LR_PATH
        try:
            print(f"\nAttempting to load BSD100 Test Set from {data_path}...")
            if data_path.exists():
                # Load all PNG images from the validation folder
                image_files = sorted(list(data_path.glob("*.png")))
                print(f"Found {len(image_files)} images in validation dataset folder.")
                
                if len(image_files) > 0:
                    for img_path in image_files:
                        img = Image.open(img_path).convert('RGB')
                        img_tensor = ToTensor()(img)  # Converts to [0,1] range and [C, H, W]
                        self.images.append(img_tensor)
                    print(f"Successfully loaded {len(self.images)} BSD100 validation images.")
                else:
                    raise FileNotFoundError(f"No PNG images found in {data_path}")
            else:
                raise FileNotFoundError(f"Dataset path does not exist: {data_path}")
        except Exception as e:
            print(f"BSD100 Loading failed: {e}. Using simulated data.")
            print("Creating synthetic BSD100-like test dataset...")
            # Create variable-sized images to simulate BSD100
            sizes = [(256, 256), (384, 384), (512, 384), (384, 512), (512, 512)]
            self.images = [torch.rand(3, h, w) for h, w in sizes * 20]  # 100 samples
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        hr_img = self.images[idx]
        
        # Generate LR using bicubic downsampling
        hr_h, hr_w = hr_img.shape[-2:]
        lr_h = hr_h // self.scale
        lr_w = hr_w // self.scale
        
        lr_img = F.interpolate(hr_img.unsqueeze(0), size=(lr_h, lr_w), 
                               mode='bicubic', align_corners=False).squeeze(0)
        
        return lr_img, hr_img

def load_test_dataset(scale=SCALE_FACTOR):
    """
    Loads the BSD100 dataset for testing, generating LR images via bicubic downsampling
    and pairing them with the HR ground truth.
    """
    try:
        print("\nAttempting to load BSD100 (B100) Test Set via TFDS...")
        dataset = BSD100Dataset(scale=scale)
        dataloader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=1, pin_memory=True)
        print("BSD100 Test Set configured for evaluation (LR generated via Bicubic downsampling).")
        return dataloader
    except Exception as e:
        print(f"TFDS BSD100 Loading failed: {e}. Returning None.")
        return None


In [ ]:
# ----------------------------------------------------------------------
# 4. MODEL DEFINITION - Real-ESRGAN Architecture
# ----------------------------------------------------------------------

class ResidualDenseBlock(nn.Module):
    """Dense Block as used in Real-ESRGAN."""
    def __init__(self, num_filters=64, growth_channel=32):
        super(ResidualDenseBlock, self).__init__()
        self.conv1 = nn.Conv2d(num_filters, growth_channel, 3, padding=1)
        self.conv2 = nn.Conv2d(num_filters + growth_channel, growth_channel, 3, padding=1)
        self.conv3 = nn.Conv2d(num_filters + 2*growth_channel, growth_channel, 3, padding=1)
        self.conv4 = nn.Conv2d(num_filters + 3*growth_channel, growth_channel, 3, padding=1)
        self.conv5 = nn.Conv2d(num_filters + 4*growth_channel, num_filters, 3, padding=1)
        self.lrelu = nn.LeakyReLU(0.2, inplace=True)
    
    def forward(self, x):
        x1 = self.lrelu(self.conv1(x))
        x2_input = torch.cat([x, x1], dim=1)
        x2 = self.lrelu(self.conv2(x2_input))
        x3_input = torch.cat([x, x1, x2], dim=1)
        x3 = self.lrelu(self.conv3(x3_input))
        x4_input = torch.cat([x, x1, x2, x3], dim=1)
        x4 = self.lrelu(self.conv4(x4_input))
        x5_input = torch.cat([x, x1, x2, x3, x4], dim=1)
        x5 = self.conv5(x5_input)
        return x5

class ResidualInResidualDenseBlock(nn.Module):
    def __init__(self, num_filters=64, growth_channel=32):
        super().__init__()
        self.rdb1 = ResidualDenseBlock(num_filters, growth_channel)
        self.rdb2 = ResidualDenseBlock(num_filters, growth_channel)
        self.rdb3 = ResidualDenseBlock(num_filters, growth_channel)
    
    def forward(self, x):
        out = x + self.rdb1(x) * 0.2
        out = out + self.rdb2(out) * 0.2
        return out + self.rdb3(out) * 0.2

class RealESRGANGenerator(nn.Module):
    def __init__(self, scale=SCALE_FACTOR, num_rrdb=12):  # Reduced from 16 to save memory
        super().__init__()
        self.scale = scale
        self.conv_first = nn.Conv2d(3, 64, 3, padding=1)
        self.rrdb_blocks = nn.Sequential(*[ResidualInResidualDenseBlock(64, 32) for _ in range(num_rrdb)])
        self.conv_body = nn.Conv2d(64, 64, 3, padding=1)
        self.lrelu = nn.LeakyReLU(0.2, inplace=True)
        
        if scale == 4:
            self.conv_up1 = nn.Conv2d(64, 64*4, 3, padding=1)
            self.conv_up2 = nn.Conv2d(64, 64*4, 3, padding=1)
        elif scale == 2:
            self.conv_up1 = nn.Conv2d(64, 64*4, 3, padding=1)
        
        self.conv_hr = nn.Conv2d(64, 64, 3, padding=1)
        self.conv_last = nn.Conv2d(64, 3, 3, padding=1)
    
    def forward(self, x):
        feat = self.conv_first(x)
        global_res = feat
        feat = self.rrdb_blocks(feat)
        feat = self.conv_body(feat) + global_res
        
        if self.scale == 4:
            feat = F.pixel_shuffle(self.lrelu(self.conv_up1(feat)), 2)
            feat = F.pixel_shuffle(self.lrelu(self.conv_up2(feat)), 2)
        elif self.scale == 2:
            feat = F.pixel_shuffle(self.lrelu(self.conv_up1(feat)), 2)
        else:
            # Fallback for other scales
            feat = F.interpolate(feat, scale_factor=self.scale, mode='bilinear', align_corners=False)
        
        out = self.conv_last(self.lrelu(self.conv_hr(feat)))
        return torch.clamp(torch.tanh(out) * 0.58 + 0.5, 0, 1)

def create_esrgan_sr_model(scale=SCALE_FACTOR, lr_size=LR_SIZE):
    """Defines a Real-ESRGAN based Super-Resolution model structure."""
    print("\nDefining PyTorch Real-ESRGAN Super-Resolution Model...")
    
    model = RealESRGANGenerator(scale=scale)
    model = model.to(device)
    
    print("Real-ESRGAN Model defined successfully.")
    return model




In [ ]:
# ----------------------------------------------------------------------
# 5. DISCRIMINATOR (Simplified Patch Discriminator)
# ----------------------------------------------------------------------

class SimpleDiscriminator(nn.Module):
    """Simplified patch discriminator - much lighter than U-Net, uses ~90% less memory."""
    def __init__(self, input_channels=3):
        super(SimpleDiscriminator, self).__init__()
        
        # Simple sequential discriminator - no skip connections, no upsampling
        self.model = nn.Sequential(
            # Input: [B, 3, H, W]
            nn.Conv2d(input_channels, 32, 4, stride=2, padding=1),  # /2
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(32, 64, 4, stride=2, padding=1),  # /4
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(64, 128, 4, stride=2, padding=1),  # /8
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(128, 256, 4, stride=2, padding=1),  # /16
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            
            # Global average pooling and output
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(256, 1, 1),
        )
    
    def forward(self, x):
        x = self.model(x)
        # Output: [B, 1, 1, 1] -> [B, 1]
        return x.squeeze(-1).squeeze(-1)

def build_discriminator(input_channels=3):
    """Build simplified discriminator model."""
    model = SimpleDiscriminator(input_channels=input_channels)
    model = model.to(device)
    return model


In [ ]:
# ----------------------------------------------------------------------
# 5. DISCRIMINATOR (U-Net style, logits output for RaGAN)
# ----------------------------------------------------------------------

class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1):
        super(ConvBlock, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride, padding=kernel_size//2)
        self.lrelu = nn.LeakyReLU(0.2, inplace=True)
    
    def forward(self, x):
        return self.lrelu(self.conv(x))

class DownBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DownBlock, self).__init__()
        self.conv1 = ConvBlock(in_channels, out_channels)
        self.conv2 = ConvBlock(out_channels, out_channels)
        self.downsample = nn.Conv2d(out_channels, out_channels, 4, stride=2, padding=1)
        self.lrelu = nn.LeakyReLU(0.2, inplace=True)
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        skip = x
        x = self.lrelu(self.downsample(x))
        return x, skip

class UpBlock(nn.Module):
    def __init__(self, in_channels, out_channels, skip_channels):
        super(UpBlock, self).__init__()
        self.upsample = nn.UpsamplingBilinear2d(scale_factor=2)
        # After upsampling x (in_channels) and concatenating with skip (skip_channels), we have in_channels + skip_channels
        self.conv1 = ConvBlock(in_channels + skip_channels, out_channels)
        self.conv2 = ConvBlock(out_channels, out_channels)
    
    def forward(self, x, skip):
        x = self.upsample(x)
        x = torch.cat([x, skip], dim=1)
        x = self.conv1(x)
        x = self.conv2(x)
        return x

class UNetDiscriminator(nn.Module):
            """U-Net style discriminator for Real-ESRGAN (memory-optimized with reduced channels)."""
        def __init__(self, input_channels=3):
            super(UNetDiscriminator, self).__init__()
            
            # Encoder - reduced channels to save memory
            self.down1 = DownBlock(input_channels, 32)   # Reduced from 64
        self.down2 = DownBlock(32, 64)    # Reduced from 128
        self.down3 = DownBlock(64, 128)   # Reduced from 256
        self.down4 = DownBlock(128, 256)  # Reduced from 512
        
        # Bottleneck - reduced
        self.bottleneck1 = ConvBlock(256, 256)  # Reduced from 512
        self.bottleneck2 = ConvBlock(256, 256)  # Reduced from 512
        
        # Decoder - reduced channels
        # up4: input 256 (from bottleneck), skip 256 (from down4), output 128
        self.up4 = UpBlock(256, 128, skip_channels=256)
        # up3: input 128 (from up4), skip 128 (from down3), output 64
        self.up3 = UpBlock(128, 64, skip_channels=128)
        # up2: input 64 (from up3), skip 64 (from down2), output 32
        self.up2 = UpBlock(64, 32, skip_channels=64)
        # up1: input 32 (from up2), skip 32 (from down1), output 32
        self.up1 = UpBlock(32, 32, skip_channels=32)
        
        # Output
        self.conv_out = nn.Conv2d(32, 1, 1)  # Reduced from 64
        self.global_pool = nn.AdaptiveAvgPool2d(1)
    
    def forward(self, x):
        x1, s1 = self.down1(x)
        x2, s2 = self.down2(x1)
        x3, s3 = self.down3(x2)
        x4, s4 = self.down4(x3)
        
        b = self.bottleneck1(x4)
        b = self.bottleneck2(b)
        
        u3 = self.up4(b, s4)
        u2 = self.up3(u3, s3)
        u1 = self.up2(u2, s2)
        u0 = self.up1(u1, s1)
        
        logits_map = self.conv_out(u0)
        pooled = self.global_pool(logits_map)
        # AdaptiveAvgPool2d(1) outputs (batch_size, channels, 1, 1), squeeze to (batch_size, channels)
        return pooled.squeeze(-1).squeeze(-1)

def build_discriminator(input_channels=3):
    """Build simplified discriminator model."""
    model = SimpleDiscriminator(input_channels=input_channels)
    model = model.to(device)
    return model


In [ ]:
# ----------------------------------------------------------------------
# 6. PERCEPTUAL LOSS (VGG19 feature extractor)
# ----------------------------------------------------------------------

class VGGFeatureExtractor(nn.Module):
    """VGG19 feature extractor for perceptual loss."""
    def __init__(self, layer_name='features.34'):  # block5_conv4 in PyTorch VGG
        super(VGGFeatureExtractor, self).__init__()
        vgg = vgg19(pretrained=True)
        # Extract features up to the specified layer
        features = list(vgg.features)
        self.features = nn.Sequential(*features[:35])  # Up to block5_conv4
        for param in self.features.parameters():
            param.requires_grad = False
    
    def forward(self, x):
        # Input is already normalized in perceptual_loss function
        return self.features(x)

def perceptual_loss(vgg_model, y_true, y_pred):
    """Compute perceptual loss using VGG features."""
    # Convert [0,1] to VGG expected range
    # VGG19 expects input normalized with ImageNet stats
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(y_true.device)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(y_true.device)
    
    y_true_normalized = (y_true - mean) / std
    y_pred_normalized = (y_pred - mean) / std
    
    f_true = vgg_model(y_true_normalized)
    f_pred = vgg_model(y_pred_normalized)
    return torch.mean(torch.abs(f_true - f_pred))

def build_vgg_feature_extractor(layer_name='features.34'):
    """Build VGG feature extractor."""
    model = VGGFeatureExtractor(layer_name=layer_name)
    model = model.to(device)
    model.eval()
    return model


In [ ]:
# ----------------------------------------------------------------------
# 7. GAN TRAINING LOOP
# ----------------------------------------------------------------------

def compute_generator_losses(generator, discriminator, vgg, lr, hr, sr, 
                             lambda_pixel=1.0, lambda_percep=0.1, lambda_adv=0.005):
    """Compute generator losses: pixel, perceptual, and adversarial."""
    # Pixel (L1) loss
    pixel_loss = torch.mean(torch.abs(hr - sr))
    
    # Perceptual loss
    percep_loss = perceptual_loss(vgg, hr, sr)
    
    # RaGAN generator loss
    d_real = discriminator(hr)
    d_fake = discriminator(sr)
    mean_real = torch.mean(d_real)
    mean_fake = torch.mean(d_fake)
    
    # Generator wants D_fake - E[D_real] to be classified as real
    g_adv_loss = F.binary_cross_entropy_with_logits(d_fake - mean_real, torch.ones_like(d_fake))
    
    total_loss = lambda_pixel * pixel_loss + lambda_percep * percep_loss + lambda_adv * g_adv_loss
    
    return total_loss, pixel_loss, percep_loss, g_adv_loss

def compute_discriminator_loss(discriminator, hr, sr):
    """Compute RaGAN discriminator loss."""
    d_real = discriminator(hr)
    d_fake = discriminator(sr)
    mean_real = torch.mean(d_real)
    mean_fake = torch.mean(d_fake)
    
    # Discriminator: classify D_real - E[D_fake] as real, D_fake - E[D_real] as fake
    real_loss = F.binary_cross_entropy_with_logits(d_real - mean_fake, torch.ones_like(d_real))
    fake_loss = F.binary_cross_entropy_with_logits(d_fake - mean_real, torch.zeros_like(d_fake))
    
    return real_loss + fake_loss

def train_one_epoch(generator, discriminator, vgg, train_loader, 
                    g_optimizer, d_optimizer, epoch, device, 
                    lambda_pixel=1.0, lambda_percep=0.1, lambda_adv=0.005):
    """Train for one epoch."""
    generator.train()
    discriminator.train()
    
    g_losses = []
    d_losses = []
    psnr_values = []
    ssim_values = []
    
    for batch_idx, (lr, hr) in enumerate(train_loader):
        # Move tensors to device with non-blocking transfer for better performance
        lr = lr.to(device, non_blocking=True)
        hr = hr.to(device, non_blocking=True)
        
        # 1) Update Discriminator
        d_optimizer.zero_grad()
        with torch.no_grad():
            sr_d = generator(lr)
            # Detach and move to CPU temporarily to reduce GPU memory pressure
            sr_d_cpu = sr_d.detach().cpu()
            hr_cpu = hr.cpu()
        
        # Move back to GPU only when needed
        d_loss = compute_discriminator_loss(discriminator, hr用, sr_d.detach())
        d_loss.backward()
        d_optimizer.step()
        
        # Store d_loss value before clearing
        d_loss_val = d_loss.item()
        d_losses.append(d_loss_val)
        
        # Clear intermediate variables
        del d_loss, sr_d, sr_d_cpu, hr_cpu
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
        # 2) Update Generator
        g_optimizer.zero_grad()
        sr = generator(lr)
        g_total, l_pix, l_perc, l_adv = compute_generator_losses(
            generator, discriminator, vgg, lr, hr, sr,
            lambda_pixel, lambda_percep, lambda_adv
        )
        g_total.backward()
        g_optimizer.step()
        
        # Metrics (compute before clearing variables)
        with torch.no_grad():
            batch_psnr = psnr(hr, sr).item()
            batch_ssim = ssim(hr, sr).item()
            psnr_values.append(batch_psnr)
            ssim_values.append(batch_ssim)
        
        g_losses.append(g_total.item())
        
        # Clear variables to free memory
        del sr, g_total, l_pix, l_perc, l_adv
        if torch.cuda.is_available() and (batch_idx + 1) % CLEAR_CACHE_EVERY_N_BATCHES == 0:
            torch.cuda.empty_cache()
        
        if (batch_idx + 1) % 10 == 0:
            print(f'Epoch {epoch}, Batch {batch_idx+1}/{len(train_loader)}, '
                  f'G_Loss: {g_losses[-1]:.4f}, D_Loss: {d_losses[-1]:.4f}, '
                  f'PSNR: {batch_psnr:.2f}, SSIM: {batch_ssim:.4f}')
    
    return {
        'g_loss': np.mean(g_losses),
        'd_loss': np.mean(d_losses),
        'psnr': np.mean(psnr_values),
        'ssim': np.mean(ssim_values)
    }


In [ ]:
# ----------------------------------------------------------------------
# 8. EVALUATION COMPONENTS
# ----------------------------------------------------------------------

def predict_super_resolution(model, lr_image):
    """Generate super-resolved image from LR input."""
    model.eval()
    with torch.no_grad():
        lr_batched = lr_image.unsqueeze(0).to(device)
        sr_float = model(lr_batched)
        sr_float = sr_float.squeeze(0)
        sr_image_uint8 = torch.clamp(sr_float * 255.0, 0, 255).byte()
    return sr_image_uint8.cpu(), sr_float.cpu()

def plot_lr_sr_hr(lr_image, sr_image, hr_image, psnr, ssim, index, scale=SCALE_FACTOR):
    """Display LR Input, SR Output, and HR Ground Truth comparison."""
    hr_size = hr_image.shape[-1]
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Resize LR for visualization
    if len(lr_image.shape) == 3:
        lr_image = lr_image.unsqueeze(0)
    lr_display = F.interpolate(lr_image.unsqueeze(0), size=(hr_size, hr_size), 
                               mode='nearest').squeeze(0)
    lr_display = (lr_display.permute(1, 2, 0) * 255).byte().cpu().numpy()
    
    # Convert HR and SR for display
    hr_display = (hr_image.permute(1, 2, 0) * 255).byte().cpu().numpy()
    sr_display = sr_image.permute(1, 2, 0).byte().cpu().numpy()
    
    axes[0].imshow(lr_display)
    axes[0].set_title(f"Low Resolution Input (x{scale} Bicubic)", fontsize=10)
    axes[0].axis("off")
    
    axes[1].imshow(sr_display)
    axes[1].set_title(f"Real-ESRGAN Output (PSNR: {psnr:.2f} dB, SSIM: {ssim:.4f})", fontsize=10)
    axes[1].axis("off")
    
    axes[2].imshow(hr_display)
    axes[2].set_title(f"High Resolution Ground Truth", fontsize=10)
    axes[2].axis("off")
    
    plt.suptitle(f"BSD100 Test Sample {index+1}", fontsize=12)
    plt.tight_layout()
    
    filepath = f"esrgan_bsd100_test_comparison_sample_{index+1}_pytorch.png"
    plt.savefig(filepath)
    plt.close(fig)
    print(f"Plot saved to {filepath}")

def run_test_evaluation_bsd100(model, test_loader, num_samples=5):
    """Run evaluation on BSD100 dataset."""
    print(f"\n--- Starting BSD100 Test Evaluation with Real-ESRGAN ({num_samples} Samples) ---")
    
    model.eval()
    total_psnr = 0.0
    total_ssim = 0.0
    count = 0
    
    with torch.no_grad():
        for i, (lr_batch, hr_batch) in enumerate(test_loader):
            if i >= num_samples:
                break
            
            lr_img_norm = lr_batch[0]  # Normalized LR [0, 1]
            hr_img_norm = hr_batch[0]  # Normalized HR [0, 1]
            
            # Upscale the image
            sr_img_uint8, sr_img_norm = predict_super_resolution(model, lr_img_norm)
            
            # Calculate metrics
            hr_tensor = hr_img_norm.unsqueeze(0).to(device)
            sr_tensor = sr_img_norm.unsqueeze(0).to(device)
            
            current_psnr = psnr(hr_tensor, sr_tensor).item()
            current_ssim = ssim(hr_tensor, sr_tensor).item()
            
            total_psnr += current_psnr
            total_ssim += current_ssim
            count += 1
            
            # Plot results
            plot_lr_sr_hr(lr_img_norm, sr_img_uint8, hr_img_norm, current_psnr, current_ssim, i)
    
    if count > 0:
        mean_psnr = total_psnr / count
        mean_ssim = total_ssim / count
        print(f"\n--- RESULTS ON BSD100 TEST SET (First {count} Samples) ---")
        print(f"Mean PSNR: {mean_psnr:.4f} dB")
        print(f"Mean SSIM: {mean_ssim:.4f}")
    else:
        print("No BSD100 test samples were available for evaluation.")
    
    print("--- BSD100 Test Evaluation Complete. ---")


In [ ]:
# ----------------------------------------------------------------------
# 9. MAIN TRAINING AND EVALUATION SCRIPT
# ----------------------------------------------------------------------

if __name__ == '__main__':
    # 1. Initialize datasets
    train_loader = load_or_simulate_dataset(
        num_samples=TRAIN_SAMPLES,
        batch_size=BATCH_SIZE,
        lr_shape=(LR_SIZE, LR_SIZE, 3),
        hr_shape=(HR_SIZE, HR_SIZE, 3)
    )
    
    test_loader = load_test_dataset()
    
    # 2. Build models
    generator = create_esrgan_sr_model(scale=SCALE_FACTOR)
    discriminator = build_discriminator(input_channels=3)
    vgg = build_vgg_feature_extractor('features.34')
    
    # 3. Optimizers with learning rate schedule
    g_optimizer = optim.Adam(generator.parameters(), lr=2e-4, betas=(0.9, 0.99))
    d_optimizer = optim.Adam(discriminator.parameters(), lr=2e-4, betas=(0.9, 0.99))
    
    # Learning rate scheduler (piecewise constant)
    g_scheduler = optim.lr_scheduler.MultiStepLR(g_optimizer, milestones=[5000], gamma=0.5)
    d_scheduler = optim.lr_scheduler.MultiStepLR(d_optimizer, milestones=[5000], gamma=0.5)
    
    print("\n" * 2)
    
    # 4. Training
    print(f"--- Starting Real-ESRGAN GAN Training ({EPOCHS} Epochs) ---")
    best_loss = float('inf')
    
    for epoch in range(1, EPOCHS + 1):
        metrics = train_one_epoch(
            generator, discriminator, vgg, train_loader,
            g_optimizer, d_optimizer, epoch, device,
            lambda_pixel=1.0, lambda_percep=0.1, lambda_adv=0.005
        )
        
        print(f"\nEpoch {epoch}/{EPOCHS} Summary:")
        print(f"  G_Loss: {metrics['g_loss']:.4f}, D_Loss: {metrics['d_loss']:.4f}")
        print(f"  PSNR: {metrics['psnr']:.2f}, SSIM: {metrics['ssim']:.4f}")
        
        # Save best model
        if metrics['g_loss'] < best_loss:
            best_loss = metrics['g_loss']
            torch.save(generator.state_dict(), 'best-esrgan-generator-pytorch.pth')
            print(f"  Saved best model (G_Loss: {best_loss:.4f})")
        
        g_scheduler.step()
        d_scheduler.step()
    
    print("--- Training Complete. Starting Evaluation. ---")
    
    # 5. Evaluation
    try:
        if test_loader:
            run_test_evaluation_bsd100(generator, test_loader, num_samples=5)
        else:
            print("\nWARNING: Could not load BSD100. Running ad-hoc visual check on training data.")
            def run_ad_hoc_evaluation_fallback(model, loader, num_samples=8):
                print(f"\n--- Starting Ad-Hoc Visual Evaluation ({num_samples} Samples, Model: Real-ESRGAN) ---")
                model.eval()
                with torch.no_grad():
                    for i, (lr_batch, _) in enumerate(loader):
                        if i >= num_samples:
                            break
                        lowres_img = lr_batch[0]
                        sr_img_uint8, _ = predict_super_resolution(model, lowres_img)
                        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
                        
                        lr_display = F.interpolate(lowres_img.unsqueeze(0), 
                                                   size=(sr_img_uint8.shape[1], sr_img_uint8.shape[2]),
                                                   mode='nearest').squeeze(0)
                        lr_display = (lr_display.permute(1, 2, 0) * 255).byte().cpu().numpy()
                        
                        axes[0].imshow(lr_display)
                        axes[0].axis("off")
                        axes[0].set_title("Low Resolution Input", fontsize=10)
                        
                        sr_display = sr_img_uint8.permute(1, 2, 0).byte().cpu().numpy()
                        axes[1].imshow(sr_display)
                        axes[1].axis("off")
                        axes[1].set_title("Real-ESRGAN Super-Resolution Output", fontsize=10)
                        
                        plt.tight_layout()
                        filepath = f"esrgan_sr_comparison_sample_{i+1}_fallback_pytorch.png"
                        plt.savefig(filepath)
                        plt.close(fig)
                        print(f"Fallback plot saved to {filepath}")
                print("--- Ad-Hoc Visual Evaluation Complete. ---")
            
            run_ad_hoc_evaluation_fallback(generator, train_loader, num_samples=8)
        
        print("\nReal-ESRGAN Model Evaluation successfully completed.")
    
    except Exception as e:
        print(f"\nFATAL ERROR: The script failed unexpectedly during evaluation.")
        print(f"Error detail: {e}")
        import traceback
        traceback.print_exc()
        sys.exit(1)
